# Sesión 12 · Redpanda → Spark → S3
Este notebook lee los lotes JSON de Aviationstack desde Kafka y los guarda en Parquet como datos Bronze.
Usa un runtime de Spark con el conector Kafka y soporte S3A compatibles con su versión.
Carga las variables de configuración en el entorno del kernel y usa las credenciales AWS del runtime.


In [ ]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("aviationstack-bronze").getOrCreate()
bootstrap_servers = os.environ["KAFKA_BOOTSTRAP_SERVERS"]
topic = os.getenv("AVIATION_KAFKA_TOPIC", "aviationstack-topic")
mechanism = os.getenv("KAFKA_SASL_MECHANISM", "SCRAM-SHA-256")
username = os.environ["KAFKA_USERNAME"]
password = os.environ["KAFKA_PASSWORD"]

# JAAS usa comillas y barras inversas como caracteres especiales.
username = username.replace(chr(92), chr(92) * 2).replace('"', chr(92) + '"')
password = password.replace(chr(92), chr(92) * 2).replace('"', chr(92) + '"')
jaas = (
    "org.apache.kafka.common.security.scram.ScramLoginModule required "
    f'username="{username}" password="{password}";'
)


In [ ]:
flights = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", mechanism)
    .option("kafka.sasl.jaas.config", jaas)
    .option("subscribe", topic)
    .option("startingOffsets", os.getenv("AVIATION_SPARK_STARTING_OFFSETS", "latest"))
    .load()
    .select(col("key").cast("string"), col("value").cast("string"),
            "topic", "partition", "offset", "timestamp")
)


In [ ]:
output_path = os.environ["AVIATION_S3_OUTPUT_PATH"]
checkpoint_path = os.environ["AVIATION_S3_CHECKPOINT_PATH"]

query = (
    flights.writeStream.format("parquet")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .start(output_path)
)
try:
    query.awaitTermination()
finally:
    query.stop()
